# Show API Connection: OpenAI

This notebook is a live, progressive demo of connecting to the OpenAI API.
Each step below builds on the previous one — by the end you'll have made a
basic call, steered it with a system prompt, and inspected exactly how many
tokens (and how much money) each call costs.

**Requirement:** a real `OPENAI_API_KEY` must be set in a `.env` file in this
folder (or the repo root). There is no mock mode — every cell below makes a
real network call to OpenAI.

## Setup (shared by every step)

We load the API key from `.env` and create one OpenAI client that every
step below will reuse. If the key is missing, this fails loudly right away
instead of letting a later cell fail with a confusing error.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Look for .env in this folder first, then fall back to the repo root.
load_dotenv()
load_dotenv(os.path.join("..", "..", ".env"))

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to a .env file before running this notebook "
        "— there is no mock mode."
    )

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"
print(f"OpenAI client ready. Using model: {MODEL}")

## Step a) Basic API call

The simplest possible thing you can do with the OpenAI API: send one user
message and print what comes back. This proves the connection works end to
end — your key is valid, the network path is open, and the model responds.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is an API?"}
    ],
)

print("Model reply:")
print(response.choices[0].message.content)

## Step b) Add a system prompt

A **system prompt** is a message with `role: "system"` that sets the
model's behavior, tone, or persona before it sees the user's question. It's
the same API call as step (a) — we're just adding one more message to the
list. Compare this reply to step (a)'s: same question, very different
voice, because the system prompt is steering it.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        # NEW: a system message added before the user message from step (a)
        {"role": "system", "content": "You are a pirate. Answer every question in pirate speak."},
        {"role": "user", "content": "In one sentence, what is an API?"},
    ],
)

print("Model reply (with system prompt):")
print(response.choices[0].message.content)

## Step c) Print token usage and estimated cost

Every OpenAI response includes a `usage` object with prompt tokens,
completion tokens, and the total. Tokens are how OpenAI bills you, so
reading this field lets you estimate the real dollar cost of a call.
Below we reuse the same system-prompt call from step (b) and print its
usage, then convert it to an estimated cost using gpt-4o-mini's published
per-token pricing.

**Note:** prices change over time — check https://openai.com/api/pricing
for current rates before trusting this number in a real budget.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a pirate. Answer every question in pirate speak."},
        {"role": "user", "content": "In one sentence, what is an API?"},
    ],
)

print("Model reply:")
print(response.choices[0].message.content)

# NEW: read and print the usage block
usage = response.usage
print("\nToken usage:")
print(f"  prompt tokens:     {usage.prompt_tokens}")
print(f"  completion tokens: {usage.completion_tokens}")
print(f"  total tokens:      {usage.total_tokens}")

# gpt-4o-mini published pricing (per 1M tokens) as of this writing — update if it changes.
PRICE_PER_1M_INPUT = 0.15
PRICE_PER_1M_OUTPUT = 0.60

estimated_cost = (
    usage.prompt_tokens / 1_000_000 * PRICE_PER_1M_INPUT
    + usage.completion_tokens / 1_000_000 * PRICE_PER_1M_OUTPUT
)
print(f"\nEstimated cost for this call: ${estimated_cost:.8f}")

## Recap

- **(a)** A minimal `chat.completions.create` call proves the API
  connection works.
- **(b)** Adding a `role: "system"` message steers behavior without
  changing the user's question.
- **(c)** The `response.usage` object tells you exactly how many tokens
  were used, which you can convert to a real dollar estimate using the
  provider's published pricing.

This notebook is a living demo — more steps (tool calling, memory, basic
RAG, etc.) can be appended later without disturbing what's already here.